# DLI vs BG Polyhedral Comparison

This notebook runs the **Deterministic Linear Inference (DLI)** and
**Backus–Gilbert (BG)** admissible-region pipelines on the same problem
using a shared set of user-controlled parameters.

Both methods build polyhedral outer approximations of the 2D property-space
admissible set using exactly $N_\theta$ sampled support-function directions,
so the resulting polyhedra can be compared face-for-face.

| Phase | Content |
|-------|---------|
| **1 (this file)** | Shared setup + DLI polyhedral path |
| **2** | BG operator-algebra path + overlay comparison figure |
| **3** | Width diagnostics, final polish |

Plan: `intervalinf/docs/agent-docs/active-plans/dli-vs-bg-polyhedral-comparison-plan.md`


In [1]:
# =============================================================================
# USER-FACING CONFIGURATION
# Edit these values to change the experiment dimensions.
# N_p must remain 2 for the 2D polyhedral comparison.
# =============================================================================
N_d = 5       # number of data
N_p = 2       # number of properties (fixed at 2 for the 2D comparison)
N_theta = 10  # number of support-function directions for polyhedral approximation


In [13]:
from intervalinf import IntervalDomain, Lebesgue, Function

from intervalinf import IntegrationConfig, ParallelConfig, LebesgueIntegrationConfig

from intervalinf.providers import NormalModesProvider, BumpFunctionProvider

from intervalinf.operators import SOLAOperator



from pygeoinf import EuclideanSpace

from pygeoinf.convex_analysis import BallSupportFunction

from pygeoinf.backus_gilbert import DualMasterCostFunction

from pygeoinf.convex_optimisation import (

    ProximalBundleMethod,

    solve_support_values,

    best_available_qp_solver,

)

from pygeoinf.subsets import HalfSpace, PolyhedralSet



import numpy as np

import seaborn as sns

import os

import time



figures_folder = 'dli_vs_bg_polyhedral_figures'

os.makedirs(figures_folder, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted', color_codes=True)

print("Imports loaded.")


Imports loaded.


## Integration and Parallel Configuration

Configure numerical integration and parallelisation settings.
With only $N_d = 5$ data this is a light problem, but we keep
the same style as `dli.ipynb` and `bg_with_errors_minkowski.ipynb`.


In [3]:
# =============================================================================
# INTEGRATION AND PARALLEL CONFIGURATION
# =============================================================================
Lebesgue_integration_cfg = LebesgueIntegrationConfig(
    inner_product=IntegrationConfig(method='simpson', n_points=500),
    dual=IntegrationConfig(method='simpson', n_points=500),
    general=IntegrationConfig(method='simpson', n_points=500),
)

sola_integration_cfg = IntegrationConfig(
    method='simpson',
    n_points=1000,
)

parallel_cfg = ParallelConfig(
    enabled=True,
    n_jobs=8,
)

print(f"SOLA integration : {sola_integration_cfg.method}, {sola_integration_cfg.n_points} pts")
print(f"Parallel         : {'enabled' if parallel_cfg.enabled else 'disabled'}, {parallel_cfg.n_jobs} jobs")


SOLA integration : simpson, 1000 pts
Parallel         : enabled, 8 jobs


## Spaces and Operators

We use the same $L^2([0,1])$ model space as `dli.ipynb`.
The data space $\mathcal{D} \cong \mathbb{R}^{N_d}$ and property space
$\mathcal{P} \cong \mathbb{R}^2$ are created from the top-level parameters.

The two property targets are bump functions centred at $x = 0.25$ and $x = 0.75$.
With $N_p = 2$, the property vector $\mathbf{p} = (p_1, p_2)$ is two-dimensional,
enabling exact 2D polyhedral plots.


In [4]:
# =============================================================================
# SPACES AND OPERATORS
# Both DLI and BG use these shared objects unchanged.
# =============================================================================
assert N_p == 2, "N_p must be 2 for the 2D polyhedral comparison."

function_domain = IntervalDomain(0, 1)
M = Lebesgue(
    0,
    function_domain,
    basis=None,
    integration_config=Lebesgue_integration_cfg,
    parallel_config=parallel_cfg,
)

D = EuclideanSpace(N_d)
P = EuclideanSpace(N_p)

x = function_domain.uniform_mesh(1000)

# Property targets: two symmetric bump functions
width = 0.3
centers = np.array([0.25, 0.75])

normal_modes_provider = NormalModesProvider(
    M,
    n_modes_range=(1, 50),
    coeff_range=(-5, 5),
    gaussian_width_percent_range=(1, 5),
    freq_range=(0.1, 20),
    random_state=2,
)

G = SOLAOperator(
    M, D,
    kernels=normal_modes_provider,
    cache_kernels=True,
    integration_config=sola_integration_cfg,
)

target_provider = BumpFunctionProvider(M, centers=centers, default_width=width)
T = SOLAOperator(
    M, P,
    kernels=target_provider,
    cache_kernels=True,
    integration_config=sola_integration_cfg,
)

print(f"M : L^2([{function_domain.a}, {function_domain.b}])")
print(f"D : R^{N_d}")
print(f"P : R^{N_p}  (targets at x={centers})")
print(f"G : M->D  (cache enabled: {G.get_cache_info()['caching_enabled']})")
print(f"T : M->P  (cache enabled: {T.get_cache_info()['caching_enabled']})")


M : L^2([0.0, 1.0])
D : R^5
P : R^2  (targets at x=[0.25 0.75])
G : M->D  (cache enabled: True)
T : M->P  (cache enabled: True)


## Synthetic Data

We use the same true model as `dli.ipynb`:
$$\bar{m}(x) = \exp\!\left(-\frac{(x-0.5)^2}{0.5^2}\right)\sin(5\pi x) + x.$$

The noise level is set via a fixed fraction of the RMS clean-data amplitude,
keeping the signal-to-noise ratio consistent as $N_d$ changes.


In [14]:
# =============================================================================

# MODEL PRIOR BALL AND DATA-CONFIDENCE BALL

# Shared by both DLI and BG.

# =============================================================================

m_0 = Function(M, evaluate_callable=lambda x: x)

model_radius = float(1.05 * M.norm(M.subtract(m_bar, m_0)))

data_radius = float(1.05 * np.linalg.norm(noise_vector))



model_prior_support = BallSupportFunction(M, m_0, model_radius)

data_error_support = BallSupportFunction(D, D.zero, data_radius)



print(f"model_radius = {model_radius:.4f}")

print(f"data_radius  = {data_radius:.4f}")


model_radius = 0.5753
data_radius  = 0.0717


## Model Prior Ball and Data-Confidence Ball

Both DLI and BG use the same deterministic sets:

$$\mathcal{B} = B_r(m_0), \qquad m_0(x) = x, \qquad r = 1.05\,\|\bar{m} - m_0\|_\mathcal{M}.$$

$$\mathcal{V} = B_\rho(\mathbf{0}), \qquad \rho = 1.05\,\|\text{noise}\|.$$

The 5% over-sizing ensures the true model and noise realisation are strictly
interior to their respective sets.


In [6]:
# =============================================================================
# MODEL PRIOR BALL AND DATA-CONFIDENCE BALL
# Shared by both DLI and BG.
# =============================================================================
m_0 = Function(M, evaluate_callable=lambda x: x)
model_radius = 1.05 * M.norm(M.subtract(m_bar, m_0))
data_radius  = 1.05 * np.linalg.norm(noise_vector)

model_prior_support = BallSupportFunction(M, m_0, model_radius)
data_error_support  = BallSupportFunction(D, D.zero, data_radius)

print(f"model_radius = {model_radius:.4f}")
print(f"data_radius  = {data_radius:.4f}")


model_radius = 0.5753
data_radius  = 0.0717


In [7]:
# =============================================================================
# VALIDATION: SHARED SETUP
# =============================================================================
assert D.dim == N_d, f"Data-space dim: got {D.dim}, expected {N_d}"
assert P.dim == N_p, f"Property-space dim: got {P.dim}, expected {N_p}"
assert G.domain == M and G.codomain == D, "G operator domain/codomain mismatch"
assert T.domain == M and T.codomain == P, "T operator domain/codomain mismatch"
assert model_radius > 0, "model_radius must be positive"
assert data_radius  > 0, "data_radius must be positive"

# m_bar must lie inside the model prior ball
_dist_m = M.norm(M.subtract(m_bar, m_0))
assert _dist_m <= model_radius + 1e-10, (
    f"m_bar not inside model prior ball: dist={_dist_m:.6f} > r={model_radius:.6f}"
)

# noise_vector must lie inside the data-confidence ball
_dist_d = np.linalg.norm(noise_vector)
assert _dist_d <= data_radius + 1e-10, (
    f"noise not inside data ball: ||noise||={_dist_d:.6f} > rho={data_radius:.6f}"
)

print("PASS: shared setup validation")
print(f"  D.dim={D.dim}, P.dim={P.dim}, N_theta={N_theta}")
print(f"  model prior  : {model_prior_support.__class__.__name__}  r={model_radius:.4f}")
print(f"  data confid. : {data_error_support.__class__.__name__}   rho={data_radius:.4f}")
print(f"  m_bar inside model prior ball : True  (dist={_dist_m:.4f} <= r={model_radius:.4f})")
print(f"  noise inside data ball        : True  (||e||={_dist_d:.4f} <= rho={data_radius:.4f})")


PASS: shared setup validation
  D.dim=5, P.dim=2, N_theta=10
  model prior  : BallSupportFunction  r=0.5753
  data confid. : BallSupportFunction   rho=0.0717
  m_bar inside model prior ball : True  (dist=0.5479 <= r=0.5753)
  noise inside data ball        : True  (||e||=0.0683 <= rho=0.0717)


## DLI Pathway — Property-Space Polyhedral Approximation

### Theory

The DLI admissible set is

$$\mathcal{U}_\text{DLI} = \mathcal{T}\!\left(\mathcal{B} \cap G^{-1}(\tilde{\mathbf{d}} - \mathcal{V})\right).$$

Its support function satisfies the **master dual equation**:

$$h_{\mathcal{U}}(q) = \inf_{\lambda \in \mathcal{D}}
\left\{ \langle \lambda,\, \tilde{\mathbf{d}} \rangle_{\mathcal{D}}
 + \sigma_{\mathcal{B}}\!\bigl(\mathcal{T}^* q - G^* \lambda\bigr)
 + \sigma_{\mathcal{V}}(-\lambda) \right\}.$$

For each of the $N_\theta$ sampled directions $q_i = (\cos\theta_i,\,\sin\theta_i)$
the infimum is solved by the **proximal bundle method**, yielding the support
value $h_i = h_{\mathcal{U}}(q_i)$.  The polyhedral outer approximation is then

$$\mathcal{U}_\text{DLI} \subseteq
\bigcap_{i=1}^{N_\theta} \{p \in \mathcal{P} : \langle q_i, p\rangle \le h_i\}.$$


In [9]:
# =============================================================================
# DLI: COST FUNCTION AND BUNDLE SOLVER
# =============================================================================
dli_cost = DualMasterCostFunction(
    D,
    P,
    M,
    G,
    T,
    model_prior_support,
    data_error_support,
    d_tilde,
    P.basis_vector(0),   # initial direction; overridden per solve by solve_support_values
)

dli_bundle_solver = ProximalBundleMethod(
    dli_cost,
    rho0=1.0,
    rho_factor=2.0,
    tolerance=1e-4,
    max_iterations=200,
    bundle_size=30,
    qp_solver=best_available_qp_solver(),
)

lambda0 = D.zero

print(f"DLI cost function : {dli_cost.__class__.__name__}")
print(f"DLI bundle solver : {dli_bundle_solver.__class__.__name__}")


DLI cost function : DualMasterCostFunction
DLI bundle solver : ProximalBundleMethod


In [10]:
# =============================================================================
# DLI: SOLVE SUPPORT FUNCTION IN N_theta DIRECTIONS
# Sample N_theta uniformly-spaced unit directions in R^2.
# For each direction q_i, solve the master dual equation to get h_U(q_i).
# =============================================================================
dli_angles     = np.linspace(0, 2 * np.pi, N_theta, endpoint=False)
dli_directions = [np.array([np.cos(t), np.sin(t)]) for t in dli_angles]

t0 = time.perf_counter()
dli_support_values, _, dli_diagnostics = solve_support_values(
    dli_cost,
    dli_directions,
    dli_bundle_solver,
    lambda0,
)
dli_elapsed = time.perf_counter() - t0

total_iters = sum(d.num_iterations for d in dli_diagnostics)

print(f"DLI solve complete: {dli_elapsed:.2f} s  ({N_theta} directions, {total_iters} bundle iters total)")
print(f"\nSupport values h_U(q_i):")
for i, (t, h) in enumerate(zip(dli_angles, dli_support_values)):
    print(f"  theta = {np.degrees(t):6.1f} deg   q = [{np.cos(t):+.4f}, {np.sin(t):+.4f}]   h = {h:+.6f}")


DLI solve complete: 3.01 s  (10 directions, 134 bundle iters total)

Support values h_U(q_i):
  theta =    0.0 deg   q = [+1.0000, +0.0000]   h = +1.293633
  theta =   36.0 deg   q = [+0.8090, +0.5878]   h = +1.635462
  theta =   72.0 deg   q = [+0.3090, +0.9511]   h = +1.721175
  theta =  108.0 deg   q = [-0.3090, +0.9511]   h = +1.611665
  theta =  144.0 deg   q = [-0.8090, +0.5878]   h = +1.347785
  theta =  180.0 deg   q = [-1.0000, +0.0000]   h = +0.935392
  theta =  216.0 deg   q = [-0.8090, -0.5878]   h = +0.498280
  theta =  252.0 deg   q = [-0.3090, -0.9511]   h = +0.240577
  theta =  288.0 deg   q = [+0.3090, -0.9511]   h = +0.350265
  theta =  324.0 deg   q = [+0.8090, -0.5878]   h = +0.786799


In [12]:
# =============================================================================

# DLI: ASSEMBLE POLYHEDRAL SET AND CONTAINMENT VALIDATION

# =============================================================================

dli_halfspaces = [

    HalfSpace(

        P,

        normal_vector=dli_directions[i],

        offset=dli_support_values[i],

    )

    for i in range(N_theta)

]



dli_admissible_region = PolyhedralSet(P, half_spaces=dli_halfspaces)



print(f"DLI PolyhedralSet assembled: {len(dli_halfspaces)} halfspaces")

print(f"  class : {dli_admissible_region.__class__.__name__}")



# ---- Halfspace containment check for p_bar ----

violations = [

    i for i in range(N_theta)

    if np.dot(dli_directions[i], p_bar) > dli_support_values[i] + 1e-6

]

assert not violations, (

    "p_bar violates sampled DLI halfspaces: "

    + ", ".join(

        f"i={i} theta={np.degrees(dli_angles[i]):.1f}deg"

        for i in violations

    )

)

print(f"  PASS: p_bar satisfies all {N_theta} halfspace constraints")

print(f"  p_bar in PolyhedralSet : {dli_admissible_region.is_element(p_bar)}")



# ---- Approximate axis-aligned extents from nearest sampled directions ----

_idx_px = int(np.argmin(np.abs(dli_angles - 0.0)))

_idx_py = int(np.argmin(np.abs(dli_angles - np.pi / 2)))

_idx_mx = int(np.argmin(np.abs(dli_angles - np.pi)))

_idx_my = int(np.argmin(np.abs(dli_angles - 3 * np.pi / 2)))



_h_px = dli_support_values[_idx_px]

_h_py = dli_support_values[_idx_py]

_h_mx = dli_support_values[_idx_mx]

_h_my = dli_support_values[_idx_my]



print(f"\nApproximate axis-aligned extents (nearest sampled direction):")

print(f"  p_1 range : [{-_h_mx:.4f},  {_h_px:.4f}]   width ≈ {_h_px + _h_mx:.4f}")

print(f"  p_2 range : [{-_h_my:.4f},  {_h_py:.4f}]   width ≈ {_h_py + _h_my:.4f}")

print(f"  True p_bar = [{p_bar[0]:.4f}, {p_bar[1]:.4f}]")


DLI PolyhedralSet assembled: 10 halfspaces
  class : PolyhedralSet
  PASS: p_bar satisfies all 10 halfspace constraints
  p_bar in PolyhedralSet : True

Approximate axis-aligned extents (nearest sampled direction):
  p_1 range : [-0.9354,  1.2936]   width ≈ 2.2290
  p_2 range : [-0.2406,  1.7212]   width ≈ 1.9618
  True p_bar = [-0.1363, 0.3637]
